# K-Nearest Neighbours: Training and Evaluation

This notebook trains a KNN classifier on preprocessed heart disease data and reports:
- Hyperparameter tuning via GridSearchCV
- K-Fold cross-validation
- Accuracy, Precision, Recall, F1-score, AUC-ROC
- Confusion Matrix, ROC Curve
- Optimal K visualisation

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("matplotlib") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
)

# Load preprocessed train/test data
train_df = pd.read_csv("heart_train_preprocessed.csv")
test_df = pd.read_csv("heart_test_preprocessed.csv")

X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]
X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## 1. Hyperparameter Tuning with GridSearchCV

In [ ]:
# Define hyperparameter grid
param_grid = {
    "n_neighbors": list(range(1, 31)),
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "minkowski"],
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print(f"Best CV F1-score: {grid_search.best_score_:.4f}")

## 2. Model Evaluation

In [ ]:
# Use the best model from grid search
model = grid_search.best_estimator_

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc_roc = roc_auc_score(y_test, y_proba)

print(f"Accuracy    : {accuracy:.4f}")
print(f"Precision   : {precision:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"F1-score    : {f1:.4f}")
print(f"AUC-ROC     : {auc_roc:.4f}")

## 3. K-Fold Cross-Validation

In [ ]:
# 5-fold and 10-fold cross-validation on the tuned model
for k in [5, 10]:
    cv_scores = cross_val_score(model, X_train, y_train, cv=k, scoring="f1")
    print(f"{k}-Fold CV F1-scores: {np.round(cv_scores, 4)}")
    print(f"  Mean: {cv_scores.mean():.4f}  Std: {cv_scores.std():.4f}\n")

## 4. Confusion Matrix & ROC Curve

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix=cm).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix")
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"KNN (AUC = {auc_roc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.show()

## 5. Optimal K Visualisation

In [ ]:
# Plot F1-score vs K for the best weight/metric combo
best_weights = grid_search.best_params_["weights"]
best_metric = grid_search.best_params_["metric"]

k_range = range(1, 31)
f1_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, weights=best_weights, metric=best_metric)
    scores = cross_val_score(knn, X_train, y_train, cv=5, scoring="f1")
    f1_scores.append(scores.mean())

plt.figure(figsize=(10, 5))
plt.plot(k_range, f1_scores, marker="o", markersize=4)
plt.axvline(x=grid_search.best_params_["n_neighbors"], color="r", linestyle="--", label=f"Best K = {grid_search.best_params_['n_neighbors']}")
plt.xlabel("Number of Neighbours (K)")
plt.ylabel("Mean CV F1-score")
plt.title("F1-score vs K")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()